In [1]:
import os
os.chdir("../../")#与easymlops同级目录

In [2]:
import numpy
numpy.__version__

'1.24.4'

In [3]:
import gensim
gensim.__version__

'4.3.0'

### 拆分数据

In [4]:
import pandas as pd
data=pd.read_csv("./data/demo.csv")
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### pipe构建

In [6]:
from easymlops import NLPPipeline
from easymlops.table.preprocessing import *
from easymlops.table.ensemble import *
from easymlops.nlp.preprocessing import *
from easymlops.nlp.representation import *
from easymlops.table.perfopt import *

In [7]:
nlp=NLPPipeline()
nlp.pipe(FixInput())\
   .pipe(FillNa(cols=["Name","Age"]))\
   .pipe(SelectCols(cols=["Name","Age"]))\
   .pipe(Lower(cols=["Name"]))\
   .pipe(ReplacePunctuation(cols=["Name"],symbols=""))\
   .pipe(SelectCols(cols=["Age","Name"]))\
   .pipe(Parallel([LsiTopicModel(cols=["Name"],num_topics=4),Word2VecModel(embedding_size=4,cols=["Name"])]))\
   .pipe(DropCols(cols=["Name"]))

x_test_new=nlp.fit(x_train).transform(x_test)
x_test_new.head(5)

,lsi_Name_0,lsi_Name_1,lsi_Name_2,lsi_Name_3,w2v_Name_0,w2v_Name_1,w2v_Name_2,w2v_Name_3
500,0.985461,0.006943,-0.049788,0.094199,-0.016030,0.003412,0.131879,0.227311
501,0.001774,-1.027363,-0.072602,0.017521,-0.168437,-0.080284,0.188844,0.057639
502,0.001778,-1.035647,-0.074126,0.017324,-0.168437,-0.080284,0.188844,0.057639
503,0.001211,-0.985805,-0.123092,-0.009465,-0.233626,-0.181061,0.165193,0.226518
504,0.001197,-0.968351,-0.120909,-0.007717,-0.233626,-0.181061,0.165193,0.226518


### 性能、一致性测试

In [8]:
nlp.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.3ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.core.SelectCols'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.nlp.preprocessing.Lower'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.nlp.preprocessing.ReplacePunctuation'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.core.SelectCols'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.en

### 持久化测试

In [9]:
nlp.save("ml.pkl")

In [10]:
nlp=NLPPipeline()
nlp.pipe(FixInput())\
   .pipe(FillNa(cols=["Name","Age"]))\
   .pipe(SelectCols(cols=["Name","Age"]))\
   .pipe(Lower(cols=["Name"]))\
   .pipe(ReplacePunctuation(cols=["Name"],symbols=""))\
   .pipe(SelectCols(cols=["Age","Name"]))\
   .pipe(Parallel([LsiTopicModel(cols=["Name"],num_topics=4),Word2VecModel(embedding_size=4,cols=["Name"])]))\
   .pipe(DropCols(cols=["Name"]))

nlp.load("ml.pkl")

In [11]:
nlp.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.41ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.core.SelectCols'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.nlp.preprocessing.Lower'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.nlp.preprocessing.ReplacePunctuation'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.core.SelectCols'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.e

In [12]:
nlp.transform(x_test[:5])

,lsi_Name_0,lsi_Name_1,lsi_Name_2,lsi_Name_3,w2v_Name_0,w2v_Name_1,w2v_Name_2,w2v_Name_3
500,0.985461,0.006943,-0.049788,0.094199,-0.016030,0.003412,0.131879,0.227311
501,0.001774,-1.027363,-0.072602,0.017521,-0.168437,-0.080284,0.188844,0.057639
502,0.001778,-1.035647,-0.074126,0.017324,-0.168437,-0.080284,0.188844,0.057639
503,0.001211,-0.985805,-0.123092,-0.009465,-0.233626,-0.181061,0.165193,0.226518
504,0.001197,-0.968351,-0.120909,-0.007717,-0.233626,-0.181061,0.165193,0.226518


In [13]:
x_test_new.head(5)

,lsi_Name_0,lsi_Name_1,lsi_Name_2,lsi_Name_3,w2v_Name_0,w2v_Name_1,w2v_Name_2,w2v_Name_3
500,0.985461,0.006943,-0.049788,0.094199,-0.016030,0.003412,0.131879,0.227311
501,0.001774,-1.027363,-0.072602,0.017521,-0.168437,-0.080284,0.188844,0.057639
502,0.001778,-1.035647,-0.074126,0.017324,-0.168437,-0.080284,0.188844,0.057639
503,0.001211,-0.985805,-0.123092,-0.009465,-0.233626,-0.181061,0.165193,0.226518
504,0.001197,-0.968351,-0.120909,-0.007717,-0.233626,-0.181061,0.165193,0.226518
